# Lyra 0.1 - GPU training on Google Colab

This notebook prepares and trains Lyra on a Colab CUDA GPU. **It refuses to start the configured 258M run without CUDA.** Colab `/content` is temporary; checkpoints, tokenizer, configs, logs, reports, and prepared data live under `/content/drive/MyDrive/Lyra/`.

Recommended: a GPU with **16 GiB VRAM class minimum** (T4/P100/V100 work as conservative starting profiles); L4 24 GiB or A100 40 GiB is preferable. Unknown/less-than-14-GiB GPUs are stopped. Estimated training footprint is ~7-11 GiB at context 1024, but this estimate is hardware/kernel dependent. The notebook prints actual free/total VRAM and selects a conservative microbatch/accumulation profile.

Set `REPO_URL` in the next setup cell to the Git URL for this repository. Repository has no configured remote in the development environment, so the URL cannot be inferred here.

In [ ]:
from pathlib import Path
import os, subprocess, json, shutil, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/YOUR_ACCOUNT/Lyra0.1.git"  # EDIT to your repository URL
REPO_DIR = Path('/content/Lyra0.1')
LYRA_ROOT = Path('/content/drive/MyDrive/Lyra')
for name in ('checkpoints', 'tokenizer', 'configs', 'logs', 'reports', 'datasets'):
    (LYRA_ROOT / name).mkdir(parents=True, exist_ok=True)
if 'YOUR_ACCOUNT' in REPO_URL:
    raise ValueError('Set REPO_URL to the actual Git repository URL before continuing.')
if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)
print('Persistent data:', LYRA_ROOT)

## 1. Check GPU and choose a safe execution profile

AMP is selected by the trainer: BF16 where supported, otherwise FP16 with GradScaler. Gradient checkpointing is enabled in `configs/colab.json`.

In [ ]:
import torch, json
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not found. In Colab select Runtime > Change runtime type > GPU, then rerun.')
props = torch.cuda.get_device_properties(0)
name = props.name
vram_gib = props.total_memory / 1024**3
free_gib = torch.cuda.mem_get_info()[0] / 1024**3
if vram_gib < 14:
    raise RuntimeError(f'{name} has only {vram_gib:.1f} GiB VRAM; Lyra training profile requires at least 12 GiB.')
if 'A100' in name and vram_gib >= 32:
    micro_batch, grad_accum = 2, 8
elif any(x in name for x in ('L4', 'A10', 'A100')):
    micro_batch, grad_accum = 1, 16
else:
    micro_batch, grad_accum = 1, 32
profile = {'gpu': name, 'vram_gib': round(vram_gib, 2), 'free_vram_gib': round(free_gib, 2),
           'bf16_supported': torch.cuda.is_bf16_supported(), 'amp': 'bf16' if torch.cuda.is_bf16_supported() else 'fp16',
           'micro_batch': micro_batch, 'gradient_accumulation': grad_accum,
           'effective_batch': micro_batch * grad_accum, 'sequence_length': 1024, 'gradient_checkpointing': True}
print(json.dumps(profile, indent=2))
with open('/content/drive/MyDrive/Lyra/reports/gpu_profile.json','w') as f: json.dump(profile,f,indent=2)

## 2. Install dependencies and persist the exact model config

In [ ]:
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[data,dev]"], check=True)
from pathlib import Path
shutil.copy2('configs/colab.json', LYRA_ROOT / 'configs/colab.json')
shutil.copy2('configs/debug.json', LYRA_ROOT / 'configs/debug.json')
print('Dependencies installed; configs copied to persistent Drive.')

## 3. Healthcheck, data preparation, and preflight

Training data comes from exactly one source: **Den4ikAI/russian_dialogues_2**. `scripts/prepare_data.py` downloads (or reuses) the raw corpus, cleans and deduplicates it, and writes train/validation/test splits (98/1/1) plus `dataset_stats.json`. The full corpus is used - no per-source limits and no reservoir sampling. If the download fails the build stops with an explicit error; it never falls back to another dataset. The cell below reuses prepared splits only after verifying that they were built from this single source.


In [ ]:
assert (REPO_DIR / 'training' / 'train.py').is_file()
assert (REPO_DIR / 'configs' / 'colab.json').is_file()
assert LYRA_ROOT.is_dir() and all((LYRA_ROOT / n).is_dir() for n in ('checkpoints','tokenizer','configs','logs','reports','datasets'))
probe = LYRA_ROOT / 'reports' / '.write_test'
probe.write_text('ok'); probe.unlink()
import torch, tokenizers, datasets
from lyra.config import LyraConfig
config = LyraConfig.from_json('configs/colab.json')
assert config.parameter_count() == 257991680 and config.gradient_checkpointing
print('Preflight PASS: dependencies, repository, Drive write access, and model config.')

In [ ]:
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "scripts/colab_healthcheck.py", "--config", "configs/colab.json", "--require-cuda"], check=True)

# Single training source: Den4ikAI/russian_dialogues_2 (no other dataset may be mixed in).
DATASET_REPO = "Den4ikAI/russian_dialogues_2"
DATA_DIR = LYRA_ROOT / 'datasets' / 'lyra_v01'
required_files = ('train.jsonl', 'validation.jsonl', 'test.jsonl', 'dataset_stats.json')
reuse_data = all((DATA_DIR / n).exists() for n in required_files)
if reuse_data:
    existing = json.loads((DATA_DIR / 'dataset_stats.json').read_text(encoding='utf-8-sig'))
    reuse_data = (existing.get('source_names') == ['ru_chat']
                  and existing.get('dataset', {}).get('repo') == DATASET_REPO
                  and existing.get('build_state') == 'SUCCESS'
                  and not existing.get('fallback_used'))
if not reuse_data:
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--out", str(DATA_DIR)], check=True)
else:
    print('Reusing prepared single-source data:', DATA_DIR)
(LYRA_ROOT / 'reports').mkdir(parents=True, exist_ok=True)
shutil.copy2(DATA_DIR / 'dataset_stats.json', LYRA_ROOT / 'reports' / 'dataset_stats.json')
stats = json.loads((DATA_DIR / 'dataset_stats.json').read_text(encoding='utf-8-sig'))
assert stats.get('source_names') == ['ru_chat'], stats.get('source_names')
assert stats.get('dataset', {}).get('repo') == DATASET_REPO
assert stats.get('build_state') == 'SUCCESS' and not stats.get('fallback_used')
print(json.dumps({
    'dataset': DATASET_REPO,
    'other_datasets': 'NONE',
    'original_dialogues': stats.get('original_dialogues'),
    'clean_dialogues': stats.get('clean_dialogues'),
    'removed_total': stats.get('removed_total'),
    'duplicates_removed': stats.get('duplicates_removed'),
    'splits': {k: v['conversations'] for k, v in stats.get('splits', {}).items()},
}, indent=2, ensure_ascii=False))
assert (DATA_DIR / 'train.jsonl').stat().st_size > 0
assert (DATA_DIR / 'validation.jsonl').stat().st_size > 0
assert (DATA_DIR / 'test.jsonl').stat().st_size > 0


## 4. Train/load tokenizer, then check the actual model parameter count

The tokenizer is trained only on the train split. A saved tokenizer on Drive is reused on later sessions so checkpoint fingerprints remain compatible.

In [ ]:
TOKENIZER = LYRA_ROOT / 'tokenizer' / 'tokenizer.json'
if not TOKENIZER.exists():
    subprocess.run([sys.executable, "scripts/train_tokenizer.py", "--data", str(DATA_DIR / "train.jsonl"), "--vocab-size", "16384", "--out", str(TOKENIZER)], check=True)
else:
    print('Reusing persistent tokenizer:', TOKENIZER)
subprocess.run([sys.executable, "scripts/colab_healthcheck.py", "--config", "configs/colab.json", "--require-cuda", "--instantiate", "--sanity"], check=True)
subprocess.run([sys.executable, "scripts/model_info.py", "--config", "configs/colab.json"], check=True)

## 5. Train with validation and Drive checkpoints

Run the cell again after a Colab disconnect: the pretraining command automatically detects `checkpoints/latest.pt` and resumes. Each run continues until the configured total step target (10,000), with validation and periodic Drive checkpoints. The exact explicit-resume command is included below. Checkpoints include optimizer, scheduler, AMP scaler, and RNG states. Keep the tokenizer, dataset, and config unchanged to pass strict resume compatibility checks.

In [ ]:
PRETRAIN_CKPT = LYRA_ROOT / 'checkpoints'
# Automatic resume from PRETRAIN_CKPT/latest.pt when it exists; otherwise initialize.
cmd = [
    'python', 'scripts/train.py', '--config', 'configs/colab.json',
    '--tokenizer', str(TOKENIZER), '--data', str(DATA_DIR / 'train.jsonl'),
    '--validation', str(DATA_DIR / 'validation.jsonl'), '--out', str(PRETRAIN_CKPT),
    '--batch-size', str(micro_batch), '--grad-accum', str(grad_accum),
    '--steps', '10000', '--save-every', '100', '--archive-every', '0', '--eval-every', '100'
]
if (PRETRAIN_CKPT / 'latest.pt').exists():
    cmd += ['--resume', str(PRETRAIN_CKPT / 'latest.pt')]
print('Running:', ' '.join(cmd))
with open(LYRA_ROOT / 'logs' / 'pretrain.log', 'a') as log:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line); log.flush()
    code = process.wait()
if code: raise RuntimeError(f'Training exited with code {code}')

### Optional second stage: assistant-only SFT

First execution initializes SFT from the completed pretraining checkpoint and resets the stage scheduler/cursor. If interrupted, rerun this cell: it resumes from the SFT `latest.pt` without resetting. Validation is run against the held-out validation split.

In [ ]:
SFT_CKPT = LYRA_ROOT / 'checkpoints' / 'sft'
sft_cmd = ['python','scripts/train.py','--config','configs/colab.json',
           '--tokenizer',str(TOKENIZER),'--data',str(DATA_DIR/'train.jsonl'),
           '--validation',str(DATA_DIR/'validation.jsonl'),'--stage','sft',
           '--out',str(SFT_CKPT),'--batch-size',str(micro_batch),
           '--grad-accum',str(grad_accum),'--steps','3000','--save-every','100','--archive-every','0','--eval-every','100']
if (SFT_CKPT/'latest.pt').exists():
    sft_cmd += ['--resume',str(SFT_CKPT/'latest.pt')]
else:
    sft_cmd += ['--resume',str(PRETRAIN_CKPT/'latest.pt'),'--reset-stage']
print('Running:', ' '.join(sft_cmd))
with open(LYRA_ROOT/'logs'/'sft.log','a') as log:
    process=subprocess.Popen(sft_cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end=''); log.write(line); log.flush()
    code=process.wait()
if code: raise RuntimeError(f'SFT exited with code {code}')

## Resume in a fresh Colab session

1. Select a GPU runtime and run setup, GPU check, installs, healthcheck, data, and tokenizer cells. Drive preserves the files when the runtime disconnects.
2. Run the pretraining cell again. It passes `--resume /content/drive/MyDrive/Lyra/checkpoints/latest.pt` automatically when present. Explicit equivalent:

```bash
python scripts/train.py \
  --config configs/colab.json \
  --resume /content/drive/MyDrive/Lyra/checkpoints/latest.pt \
  --tokenizer /content/drive/MyDrive/Lyra/tokenizer/tokenizer.json \
  --data /content/drive/MyDrive/Lyra/datasets/lyra_v01/train.jsonl \
  --validation /content/drive/MyDrive/Lyra/datasets/lyra_v01/validation.jsonl \
  --out /content/drive/MyDrive/Lyra/checkpoints \
  --batch-size 1 --grad-accum 16 --steps 10000
```

The GPU check cell may select different batch/accumulation values for A100. Do not change model config, tokenizer, or training data while resuming the same stage. For SFT, rerun the optional SFT cell; it will either initialize from pretrain or continue from `checkpoints/sft/latest.pt`.